# TP 8 - OSMnx route calculator (optional)

This notebook is the second part of TP 8: load the street network of Lyon
from OpenStreetMap, inspect its attributes, find the nodes closest to two
addresses, compute a route with Dijkstra and draw it on a map.

It is **not executed** in the repository because it needs an internet
connection and the optional `osmnx` / `geopandas` packages.  Install them
with:

```bash
pip install -e ".[routing]"
```

In [ ]:
# Uncomment in a fresh environment:
# %pip install osmnx geopandas folium

In [ ]:
import networkx as nx
import osmnx as ox

# Draw the figures inside the notebook.
%matplotlib inline

## 1. Load and display the network of Lyon

`graph_from_place` downloads the drivable network.  The result is a
`MultiDiGraph`: two directions are separate arcs, and parallel arcs are
possible.

In [ ]:
G = ox.graph_from_place("Lyon, France", network_type="drive")
print(type(G))
print("vertices:", G.number_of_nodes())
print("arcs:", G.number_of_edges())

In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_color="yellow",
    edge_linewidth=0.5,
    bgcolor="black",
    figsize=(10, 10),
)

## 2. Inspect the edge attributes

OSMnx stores a dictionary on every arc: length, speed, highway type, name,
geometry, etc.

In [ ]:
first_edge = next(iter(G.edges(keys=True, data=True)))
print(first_edge)

u, v = first_edge[0], first_edge[1]
print(G.get_edge_data(u, v))

The arcs can be converted to a `GeoDataFrame`, which provides interactive
maps and spatial operations.

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
print(edges.columns)
edges[["length", "highway", "name"]].head()

## 3. Renumber the vertices

OpenStreetMap identifiers are large and sparse.  For the array-based version
of Dijkstra we relabel the vertices `0, 1, 2, ...` with an explicit mapping.

In [ ]:
mapping = {node: index for index, node in enumerate(G.nodes)}
H = nx.relabel_nodes(G, mapping)
print("first identifiers:", list(G.nodes)[:5])
print("new identifiers:", list(H.nodes)[:5])

## 4. Find the nodes closest to the two addresses

`ox.geocode` returns `(latitude, longitude)`.  `nearest_nodes` expects
`X=longitude`, `Y=latitude`.

In [ ]:
origin_address = "6 Rue Plasson et Chaize, Lyon, France"
destination_address = "21 Rue Thenard, Lyon, France"

origin_lat, origin_lon = ox.geocode(origin_address)
destination_lat, destination_lon = ox.geocode(destination_address)
print(origin_lat, origin_lon)
print(destination_lat, destination_lon)

origin = ox.distance.nearest_nodes(H, X=origin_lon, Y=origin_lat)
destination = ox.distance.nearest_nodes(H, X=destination_lon, Y=destination_lat)
print("origin node:", origin)
print("destination node:", destination)

## 5. Compute and draw the route

The `dijkstra_heap` function of the repository handles the nested
`{0: {"length": ...}}` dictionaries of a `MultiDiGraph`.

In [ ]:
import time
from sda.tp08_networkx import dijkstra1, dijkstra2, dijkstra_heap

for algorithm in (dijkstra1, dijkstra2, dijkstra_heap):
    start = time.perf_counter()
    distance, route = algorithm(H, origin, destination)
    elapsed = time.perf_counter() - start
    print(f"{algorithm.__name__}: {distance:.1f} m in {elapsed:.2f} s")

print("route:", route)

In [ ]:
ox.plot_graph_route(
    H,
    route,
    route_color="red",
    route_linewidth=3,
    node_size=0,
    bgcolor="black",
    figsize=(10, 10),
)

## 6. What to remember

- OSMnx converts OpenStreetMap data into a NetworkX graph; every geographic
  attribute stays available on the vertices and arcs.
- The street graph is directed and multi-edged, so the edge weight is
  `data[0]["length"]`, not `data["length"]`.
- The same Dijkstra implementation works on the synthetic graph and on the
  real road network.
- Relabelling the vertices makes the array-based versions practical; the heap
  version does not need it but keeps the comparison fair.

The notebook is intentionally left unexecuted in the repository: the output
depends on the OpenStreetMap data at the time of the download.